# S=7 support sweep — combine seeds 42, 43 and 44 (read-only)

**Sources.**
* Seed 42: notebooks 23a/23b.
* Seed 43: notebook 27a.
* Seed 44: notebook 27b.

**Reads only.** This notebook reads saved predictions; it does not train or re-evaluate, so a CPU runtime is
enough. Unfinished runs show as "not yet available" and are left out of the averages, so it can be rerun at any
time.

**Outputs.**
1. Availability of every (condition, model, seed).
2. Exact accuracy by condition, model and test k: mean ± std over seeds, plus each seed's value.
3. One panel per training support: accuracy vs test k (k=1 = singletons).
   * Thick line = mean over seeds, band = min–max, thin lines = individual seeds.
   * Grey = trained k; open markers = held out.
4. Singleton / +1 step / +2 step accuracy vs longest trained k (line = mean, dots = seeds).
5. A compact summary table of the held-out numbers per support and model.

**Send back `S7_Sweep_MultiSeed_Review.ipynb`.**


In [ ]:
# --- 2. Bootstrap: locate or clone the PRIVATE repository (credentials are never printed or stored) ---
import base64, importlib.util, os, pathlib, subprocess, sys

REPO_URL = "https://github.com/zhexuandliu/Decompositionality.git"
REPO_BRANCH = "main"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

def _find_repo():
    candidates = [pathlib.Path("/content/Decompositionality"), pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    for p in candidates:
        if (p / "src" / "decompositionality").is_dir():
            return p.resolve()
    return None

REPO_DIR = _find_repo()
if REPO_DIR is None:
    token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
    if not token:
        import getpass
        token = getpass.getpass("GitHub token with read access to the private repo (input hidden): ")
    # Pass the credential through environment-scoped git config (not the URL, not the command line,
    # not persisted in .git/config).
    _hdr = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
    _env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_CONFIG_COUNT="1",
                GIT_CONFIG_KEY_0="http.https://github.com/.extraheader", GIT_CONFIG_VALUE_0=_hdr)
    REPO_DIR = pathlib.Path("/content/Decompositionality" if IN_COLAB else "Decompositionality").resolve()
    _r = subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
                        env=_env, capture_output=True, text=True)
    del token, _hdr, _env
    if _r.returncode != 0:
        raise RuntimeError("git clone failed (check the GITHUB_TOKEN secret / repo access):\n" + _r.stderr[-2000:])

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
for _mod in ("numpy", "pandas", "matplotlib", "torch"):
    if importlib.util.find_spec(_mod) is None:  # Colab already ships these; never reinstall torch/CUDA here
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _mod], check=True)
REPO_COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Repository:", REPO_DIR, "| commit:", REPO_COMMIT or "unknown")


In [ ]:
# --- CONFIGURATION -------------------------------------------------------------------------------------------------
MODE = "full"
SEEDS = [42, 43, 44]
SUBDIR = "S7_support_sweep_20k_v1"
USE_DRIVE = IN_COLAB
RESULTS_ROOT = os.environ.get("DECOMP_RESULTS_ROOT") or (
    "/content/drive/MyDrive/decompositionality_results" if USE_DRIVE else str(REPO_DIR / "results"))
# -----------------------------------------------------------------------------------------------------------------
if USE_DRIVE and IN_COLAB and RESULTS_ROOT.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
import pandas as pd
from IPython.display import display
pd.set_option("display.max_rows", 500); pd.set_option("display.max_columns", 60); pd.set_option("display.width", 250)
from decompositionality import longk as LK, longk_exp as LX


In [ ]:
# --- Combined report (reads saved artifacts only) -------------------------------------------------------------------
rp = LX.multiseed_report(RESULTS_ROOT, MODE, SEEDS, LK.S7_SWEEP, SUBDIR)
print("REVIEW FILE TO SEND BACK:", rp)


In [ ]:
# --- Compact held-out summary: mean [min, max] over seeds per support x model ---------------------------------------
df = LX.per_k_accuracy(RESULTS_ROOT, MODE, SEEDS, LK.S7_SWEEP, SUBDIR)
ok = df[df.status == "available"]
fmt = lambda s: f"{s.mean():.3f} [{s.min():.3f}, {s.max():.3f}] (n={s.count()})"
held = ok[ok.role != "trained"]
summary = held.pivot_table(index=["condition", "model"], columns="role", values="exact_acc", aggfunc=fmt)
trained = ok[ok.role == "trained"].groupby(["condition", "model"]).exact_acc.agg(lambda s: f"{s.min():.4f}")
summary["trained k (min over k, seeds)"] = trained
cols = [c for c in ["trained k (min over k, seeds)", "singleton", "+1 step", "+2 step", "+3 step", "+4 step", "+5 step"]
        if c in summary.columns]
summary = summary[cols].reset_index()
display(summary)
summary.to_csv(LX.family_root(RESULTS_ROOT, MODE, SUBDIR) / "report" / "multiseed" / "tables" / "heldout_summary.csv", index=False)
if IN_COLAB and rp:
    from google.colab import files
    files.download(str(rp))
